# подготовка датасета

In [6]:
import asyncio
import json
import logging

import pandas as pd
import random
from numpy.random import RandomState
from pydantic import TypeAdapter

from src.schemas import (
    PromptLogprob,
    PtbScenario,
    PtbScenarioRes,
    TokenEntropy,
    WordInfo,
    WordInfoRes,
    ReadingComprehensionItem,
    TokenEntropy,
)

logger = logging.getLogger("research")

In [7]:
df = pd.read_json("../MuSeRC/train.jsonl", lines=True)

small_list = df.to_dict(orient="records")
ta = TypeAdapter(list[ReadingComprehensionItem])
items = ta.validate_python(small_list)

In [8]:
len(items)

500

## Обработка

1. Выбираем вопросы, где хотя бы 1 ответ правильный
2. Выбираем случайно один правильный ответ и записываем его как эталон
3. Если остались ещё правильные ответы - записываем их как генерацию
4. Берём сколько же неправильных ответов, сколько оставалось правильных - записываем их как генерации провальные

In [10]:
from src.muserc_split import flat_generation_records

In [11]:
rng = random.Random(42)

rows = flat_generation_records(items, rng)

In [12]:
eval_set = pd.DataFrame(rows)

In [13]:
eval_set.index.name = "id"

In [14]:
eval_set_clean = eval_set.drop([4623,4624,4625,4626])
eval_set_clean = eval_set_clean.drop([4914,4915])

In [15]:
eval_set_clean["label"].value_counts()

label
1    2482
0    2432
Name: count, dtype: int64

In [16]:
eval_set.to_csv("../data/dataset_QA.csv", index_label="id")

## подгрузим в clearml

### настройка

In [38]:
import plotly.express as px
import plotly.graph_objects as go

In [25]:
import plotly.io as pio
pio.renderers.default = "iframe" # или "notebook_connected"

In [17]:
import os
os.environ["CLEARML_CONFIG_FILE"] = "../megallm.conf"

import clearml
from clearml import Dataset

In [ ]:
def log_git_info_to_clearml(task, repo_path='.'):
    try:
        # Ищем корень репозитория (пойдет вверх по папкам, пока не найдет .git)
        repo = git.Repo(repo_path, search_parent_directories=True)
        
        # Получаем хэш текущего коммита
        commit_hash = repo.head.commit.hexsha
        
        # Получаем активную ветку (обрабатываем ошибку, если у нас detached HEAD)
        try:
            branch_name = repo.active_branch.name
        except TypeError:
            branch_name = "detached_HEAD"
            
        # Получаем URL удаленного репозитория (обычно origin)
        try:
            remote_url = list(repo.remote('origin').urls)[0]
        except ValueError:
            remote_url = "local_only"
            
        # Получаем незакоммиченные изменения (diff)
        diff_text = repo.git.diff()

        # Пытаемся автоматически получить имя ноутбука
        try:
            # Возвращает объект пути (pathlib.Path) к текущему ноутбуку
            notebook_path = ipynbname.path()
            # Извлекаем само имя файла (например, 'experiment.ipynb')
            current_file_name = notebook_path.name 
            # Также можно получить путь относительно корня проекта для working_dir
            working_dir = str(notebook_path.parent)
        except FileNotFoundError:
            # Фолбек на случай, если скрипт запущен не в классическом Jupyter
            current_file_name = "unknown_notebook.ipynb"
            working_dir = "."
            print("⚠️ Не удалось автоматически определить имя ноутбука.")
        
        # 2. Передаем все собранные данные в ClearML
        task.set_script(
            repository=remote_url,
            branch=branch_name,
            commit=commit_hash,
            diff=diff_text,
            working_dir=working_dir,
            entry_point=current_file_name # <--- Имя подставилось автоматически!
        )
        print(f"✅ Git залогирован: Ветка '{branch_name}', Коммит {commit_hash[:7]}")
        
    except git.exc.InvalidGitRepositoryError:
        print("❌ Ошибка: Текущая директория не является Git-репозиторием.")

### Загрузка

In [18]:
# 1. Создаем пустой объект датасета
dataset = Dataset.create(
    dataset_name="MuSeRC_QA_eval",
    dataset_project="RAG_Metrics"
)

ClearML results page: http://100.110.148.39:8080/projects/00d072d80bef4b5bb2e0396aa0f1826e/experiments/08e459b3139a47c8a531c8381294dec6/output/log
ClearML dataset page: http://100.110.148.39:8080/datasets/simple/00d072d80bef4b5bb2e0396aa0f1826e/experiments/08e459b3139a47c8a531c8381294dec6


In [19]:
# 2. Добавляем файлы или целые папки
# Можно добавить локальную папку или отдельные файлы
dataset.add_files(path="../data/muserc_qa_eval/")

Generating SHA2 hash for 2 files


100%|████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████| 2/2 [00:00<00:00, 1334.70it/s]

Hash generation completed


2

In [20]:
logger = dataset.get_logger()

In [49]:
class_counts = eval_set_clean['label'].value_counts().sort_index()
custom_class_names = ['Не Правильные', 'Правильные']

# 3. Строим график через graph_objects
fig = go.Figure(data=[
    go.Bar(
        x=custom_class_names,
        y=class_counts.values,
        text=class_counts.values, # Текст со значениями
        textposition='auto',             # Автоматическое позиционирование текста (внутри или снаружи)
        textfont=dict(size=32),
        marker_color=['#636EFA', '#EF553B', '#00CC96'], # Задаем цвета для каждого столбца вручную
        marker_line_color='rgb(8,48,107)', # Цвет обводки
        marker_line_width=1.5,             # Толщина обводки
        opacity=0.8                        # Прозрачность столбцов
    )
])

# 4. Детальная настройка макета (Layout)
fig.update_layout(
    title={
        'text': 'Распределение ответов',
        'y': 0.9,
        'x': 0.5,
        'xanchor': 'center',
        'yanchor': 'top'
    },
    font=dict(
        family="Arial, sans-serif",
        size=16, # <-- БАЗОВЫЙ размер шрифта для графика
        color="black"
    ),
    yaxis_title='Количество ответов',
    plot_bgcolor='rgba(240, 240, 240, 0.5)', # Цвет фона области графика
    paper_bgcolor='white',                   # Цвет фона вокруг графика
    margin=dict(l=40, r=40, t=60, b=40)      # Отступы
)

ValueError: Mime type rendering requires nbformat>=4.2.0 but it is not installed

Figure({
    'data': [{'marker': {'color': ['#636EFA', '#EF553B', '#00CC96'], 'line': {'color': 'rgb(8,48,107)', 'width': 1.5}},
              'opacity': 0.8,
              'text': {'bdata': 'AAAAAAAAo0AAAAAAAGSjQA==', 'dtype': 'f8'},
              'textfont': {'size': 32},
              'textposition': 'auto',
              'type': 'bar',
              'x': [Не Правильные, Правильные],
              'y': {'bdata': 'gAmyCQ==', 'dtype': 'i2'}}],
    'layout': {'font': {'color': 'black', 'family': 'Arial, sans-serif', 'size': 16},
               'margin': {'b': 40, 'l': 40, 'r': 40, 't': 60},
               'paper_bgcolor': 'white',
               'plot_bgcolor': 'rgba(240, 240, 240, 0.5)',
               'template': '...',
               'title': {'text': 'Распределение ответов', 'x': 0.5, 'xanchor': 'center', 'y': 0.9, 'yanchor': 'top'},
               'yaxis': {'title': {'text': 'Количество ответов'}}}
})

In [50]:
logger.report_plotly(
    title="EDA",
    series="Распределение ответов",
    figure=fig,
)
    

In [27]:
class_dist = pd.DataFrame(eval_set_clean["label"].value_counts())

In [30]:
logger.report_table(
    title='EDA',
    series='Баланс Классов',
    table_plot=class_dist
)

In [51]:
# 3. (Опционально) Добавляем метаданные или теги
dataset.add_tags(["full", "QA", "Human"])
logger.report_text(
"""
dataset_QA.csv -- id контекста; вопрос; эталон; "сгенерированный" ответ
id контекста обязательно преобразовывать к str.
passages.json -- id контекста; контекст
"""
)


dataset_QA.csv -- id контекста; вопрос; эталон; "сгенерированный" ответ
id контекста обязательно преобразовывать к str.
passages.json -- id контекста; контекст



In [53]:
logger.report_text(
"""
процесс обработки

1. Выбираем вопросы, где хотя бы 1 ответ правильный
2. Выбираем случайно один правильный ответ и записываем его как эталон
3. Если остались ещё правильные ответы - записываем их как генерацию
4. Берём сколько же неправильных ответов, сколько оставалось правильных - записываем их как генерации провальные
"""
)


процесс обработки

1. Выбираем вопросы, где хотя бы 1 ответ правильный
2. Выбираем случайно один правильный ответ и записываем его как эталон
3. Если остались ещё правильные ответы - записываем их как генерацию
4. Берём сколько же неправильных ответов, сколько оставалось правильных - записываем их как генерации провальные



In [52]:
# 4. Загружаем данные на сервер
dataset.upload()

Uploading dataset changes (2 files compressed to 1.71 MiB) to http://100.110.148.39:8081
File compression and upload completed: total size 1.71 MiB, 1 chunk(s) stored (average size 1.71 MiB)


In [54]:
dataset.set_description("Создан на основе MuSeRC датасета, содержит вопрос, эталонный ответ и генерацию.")

In [55]:
# 5. Финализируем (делает версию неизменяемой)
dataset.finalize()

True